# 02 · Construir train, validation y test

Este notebook convierte las candidatas auditadas de Label Studio en un conjunto fijo de 1500 pares `noticia → reference_claim`, cada uno con su `tema`, con 1200 filas de entrenamiento, 150 de validación y 150 de prueba.

La selección y los splits son deterministas. No se vuelve a ejecutar después de entrenar salvo que exista una razón metodológica documentada.

## 1. Cargar las candidatas auditadas

El notebook anterior produce este archivo. Solo contiene filas aptas, con título, cuerpo, `reference_claim` y `tema`; aun así se vuelven a validar longitud y duplicados exactos antes de cerrar el conjunto.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

INPUT_PATH = ROOT / 'artifacts' / 'labelstudio' / 'processed' / 'current_export' / 'reference_claims_candidates.csv'
DATASET_NAME = 'claims_1500'
FINAL_DIR = ROOT / 'artifacts' / 'sem1' / DATASET_NAME
TARGET_ROWS = 1500
SPLIT_SIZES = {'train': 1200, 'validation': 150, 'test': 150}
# Estas semillas fijan la selección y los splits para poder reconstruirlos después.
SELECTION_SEED = 'sem1_reference_selection_1500_v1'
SPLIT_SEED = 'sem1_fixed_split_1500_v1'
OVERWRITE_EXISTING = False

if sum(SPLIT_SIZES.values()) != TARGET_ROWS:
    raise ValueError('Los tamaños de split deben sumar TARGET_ROWS.')
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f'No existe el CSV de candidatas: {INPUT_PATH}')

print(f'Entrada: {INPUT_PATH.relative_to(ROOT)}')
print({'dataset': DATASET_NAME, 'diseño_fijo': SPLIT_SIZES})

## 2. Validar y deduplicar

Se excluyen filas sin datos esenciales o sin `tema`, con claims fuera de 5–80 palabras o con `source_text`/claim exacto repetido. La elección entre duplicados y el muestreo de 1500 usan hashes con semilla fija.

In [ ]:
import csv
import hashlib
import json
import re
from collections import Counter

REQUIRED_COLUMNS = {'record_id', 'title', 'body', 'reference_claim_normalized', 'tema_normalized'}

def clean_text(value):
    return re.sub(r'\s+', ' ', str(value or '')).strip()

def digest(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()[:16]

def stable_rank(seed, record_id):
    return hashlib.sha256(f'{seed}:{record_id}'.encode('utf-8')).hexdigest()

with INPUT_PATH.open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    input_columns = reader.fieldnames or []
    input_rows = list(reader)

missing_columns = sorted(REQUIRED_COLUMNS.difference(input_columns))
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {', '.join(missing_columns)}")

valid_rows, invalid_rows = [], []
for row in input_rows:
    record_id = clean_text(row.get('record_id'))
    title = clean_text(row.get('title'))
    body = clean_text(row.get('body'))
    reference_claim = clean_text(row.get('reference_claim_normalized'))
    tema = clean_text(row.get('tema_normalized'))
    claim_words = len(re.findall(r'\b[\wÀ-ÿ]+\b', reference_claim, flags=re.UNICODE))
    flags = []
    if not record_id:
        flags.append('SIN_RECORD_ID')
    if not title:
        flags.append('SIN_TITULO')
    if not body:
        flags.append('SIN_CUERPO')
    if not 5 <= claim_words <= 80:
        flags.append('LONGITUD_CLAIM_INVALIDA')
    if not tema:
        flags.append('SIN_TEMA')
    if flags:
        invalid_rows.append({'record_id': record_id, 'reason': '|'.join(flags)})
        continue
    source_text = '\n\n'.join((title, body))
    normalized = dict(row)
    normalized.update({
        'record_id': record_id, 'title': title, 'body': body,
        'source_text': source_text, 'reference_claim': reference_claim, 'tema': tema,
        'source_hash': digest(source_text), 'claim_hash': digest(reference_claim.casefold()),
        'source_char_count': len(source_text), 'claim_word_count': claim_words,
        'selection_rank': stable_rank(SELECTION_SEED, record_id),
    })
    valid_rows.append(normalized)

selected_pool, duplicate_rows = [], []
seen_record_ids, seen_sources, seen_claims = set(), set(), set()
for row in sorted(valid_rows, key=lambda item: item['selection_rank']):
    duplicate_reason = ''
    if row['record_id'] in seen_record_ids:
        duplicate_reason = 'RECORD_ID_DUPLICADO'
    elif row['source_hash'] in seen_sources:
        duplicate_reason = 'SOURCE_TEXT_DUPLICADO'
    elif row['claim_hash'] in seen_claims:
        duplicate_reason = 'REFERENCE_CLAIM_DUPLICADO'
    if duplicate_reason:
        duplicate_rows.append({'record_id': row['record_id'], 'reason': duplicate_reason})
        continue
    seen_record_ids.add(row['record_id'])
    seen_sources.add(row['source_hash'])
    seen_claims.add(row['claim_hash'])
    selected_pool.append(row)

if len(selected_pool) < TARGET_ROWS:
    raise ValueError(f'Solo hay {len(selected_pool)} pares únicos y válidos; se requieren {TARGET_ROWS}.')

reference_rows = selected_pool[:TARGET_ROWS]
reserve_rows = selected_pool[TARGET_ROWS:]
print(f'Filas de entrada: {len(input_rows)}')
print(f'Válidas antes de deduplicar: {len(valid_rows)}')
print(f'Duplicados exactos excluidos: {len(duplicate_rows)}')
print(f'Conjunto de referencia: {len(reference_rows)}')
print(f'Reserva única: {len(reserve_rows)}')

## 3. Fijar los splits

Los 1500 `record_id` ya elegidos se ordenan con otra semilla fija y se reparten por tema, para que cada uno conserve su proporción en train, validation y test. Cada fila pertenece a exactamente un split; no hay texto fuente ni claim exacto repetido entre train, validation y test.

In [ ]:
# El orden estable evita que volver a ejecutar el notebook cambie train, validation y test.
ordered_for_split = sorted(reference_rows, key=lambda row: stable_rank(SPLIT_SEED, row['record_id']))

def allocate(total, quotas, capacity):
    # Reparte por restos mayores para sumar exactamente total sin exceder las filas de cada tema.
    counts = {tema: min(int(quota), capacity[tema]) for tema, quota in quotas.items()}
    by_remainder = sorted(quotas, key=lambda tema: (int(quotas[tema]) - quotas[tema], tema))
    pending = total - sum(counts.values())
    while pending:
        for tema in by_remainder:
            if pending and counts[tema] < capacity[tema]:
                counts[tema] += 1
                pending -= 1
    return counts

# Estratificar evita que un tema poco frecuente quede casi ausente de la prueba y distorsione el F1 macro.
tema_sizes = Counter(row['tema'] for row in ordered_for_split)
validation_by_tema = allocate(
    SPLIT_SIZES['validation'],
    {tema: size * SPLIT_SIZES['validation'] / TARGET_ROWS for tema, size in tema_sizes.items()},
    tema_sizes,
)
test_by_tema = allocate(
    SPLIT_SIZES['test'],
    {tema: size * SPLIT_SIZES['test'] / TARGET_ROWS for tema, size in tema_sizes.items()},
    {tema: size - validation_by_tema[tema] for tema, size in tema_sizes.items()},
)

seen_by_tema = Counter()
for row in ordered_for_split:
    tema = row['tema']
    position = seen_by_tema[tema]
    seen_by_tema[tema] += 1
    train_size = tema_sizes[tema] - validation_by_tema[tema] - test_by_tema[tema]
    row['split'] = 'train' if position < train_size else 'validation' if position < train_size + validation_by_tema[tema] else 'test'

missing_in_eval = sorted(tema for tema in tema_sizes if not validation_by_tema[tema] or not test_by_tema[tema])
if missing_in_eval:
    print('Aviso: temas sin filas en validation o test:', ', '.join(missing_in_eval))

split_counts = Counter(row['split'] for row in ordered_for_split)
if dict(split_counts) != SPLIT_SIZES:
    raise AssertionError(f'Splits incorrectos: {dict(split_counts)}')
if len({row['record_id'] for row in ordered_for_split}) != TARGET_ROWS:
    raise AssertionError('Hay record_id repetidos en el conjunto final.')
if len({row['source_hash'] for row in ordered_for_split}) != TARGET_ROWS:
    raise AssertionError('Hay source_text repetido en el conjunto final.')
if len({row['claim_hash'] for row in ordered_for_split}) != TARGET_ROWS:
    raise AssertionError('Hay reference_claim repetido en el conjunto final.')

print('Splits fijos:', dict(split_counts))
print('Distribución por tema:', {split: dict(sorted(Counter(row['tema'] for row in ordered_for_split if row['split'] == split).items())) for split in SPLIT_SIZES})
print('Distribución por país:', {split: dict(sorted(Counter(row.get('country', '') for row in ordered_for_split if row['split'] == split).items())) for split in SPLIT_SIZES})

## 4. Escribir los CSV finales

Los archivos se guardan bajo `artifacts/sem1/claims_1500/`. Por defecto el notebook no sobrescribe un conjunto ya creado: cambia `OVERWRITE_EXISTING` a `True` solo antes de entrenar y con una razón metodológica clara.

In [ ]:
FINAL_COLUMNS = [
    'record_id', 'country', 'requested_country', 'published_at', 'source_name', 'canonical_url',
    'title', 'subtitle_or_bajada', 'body', 'source_text', 'reference_claim', 'tema',
    'source_hash', 'claim_hash', 'source_char_count', 'claim_word_count', 'split',
]
MANIFEST_COLUMNS = [
    'record_id', 'country', 'requested_country', 'tema', 'source_hash', 'claim_hash',
    'source_char_count', 'claim_word_count', 'selection_rank', 'split',
]
OUTPUTS = {
    'reference': FINAL_DIR / 'reference_claims.csv',
    'train': FINAL_DIR / 'train.csv',
    'validation': FINAL_DIR / 'validation.csv',
    'test': FINAL_DIR / 'test.csv',
    'manifest': FINAL_DIR / f'manifest_final_{TARGET_ROWS}.csv',
    'reserve': FINAL_DIR / 'reserve_candidates.csv',
    'summary': FINAL_DIR / 'split_summary.json',
}
existing_outputs = [path for path in OUTPUTS.values() if path.exists()]
if existing_outputs and not OVERWRITE_EXISTING:
    raise FileExistsError('Ya existe un conjunto final. No se sobrescribe: ' + ', '.join(path.name for path in existing_outputs))
FINAL_DIR.mkdir(parents=True, exist_ok=True)

def write_csv(path, rows, columns):
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

write_csv(OUTPUTS['reference'], ordered_for_split, FINAL_COLUMNS)
for split in SPLIT_SIZES:
    write_csv(OUTPUTS[split], [row for row in ordered_for_split if row['split'] == split], FINAL_COLUMNS)
write_csv(OUTPUTS['manifest'], ordered_for_split, MANIFEST_COLUMNS)
write_csv(OUTPUTS['reserve'], reserve_rows, FINAL_COLUMNS)

summary = {
    'dataset_name': DATASET_NAME, 'target_rows': TARGET_ROWS, 'split_specification': SPLIT_SIZES,
    'input_candidates': len(input_rows), 'valid_before_deduplication': len(valid_rows),
    'excluded_invalid': len(invalid_rows), 'excluded_exact_duplicates': len(duplicate_rows),
    'reference_rows': len(ordered_for_split), 'reserve_rows': len(reserve_rows),
    'split_counts': dict(split_counts), 'selection_seed': SELECTION_SEED, 'split_seed': SPLIT_SEED,
    'tema_counts': dict(sorted(Counter(row['tema'] for row in ordered_for_split).items())),
}
OUTPUTS['summary'].write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

for name, path in OUTPUTS.items():
    print(f'{name}: {path.relative_to(ROOT)}')

## Resultado

`train.csv`, `validation.csv` y `test.csv` son los únicos archivos que deben alimentar entrenamiento, selección de checkpoint y evaluación final, respectivamente. El conjunto de prueba no debe usarse para tomar decisiones de entrenamiento.